## Problem Statement

### Business Context

GlobalEdge Brokerage is a mid-sized brokerage firm operating across multiple countries, serving thousands of retail and institutional clients through a network of equity brokers. Brokers handle investment recommendations, portfolio reviews, and market advisory discussions across equity, forex, commodity, crypto, and international stock markets. Each broker manages hundreds of clients and is expected to stay updated with overnight market developments before the market opens.

Every morning, brokers must review large volumes of financial news, stock-price movements, earnings discussions, analyst commentary, and SEC regulatory filings within a very limited preparation window. In practice, brokers can only read a small portion of the available information before their first client calls begin. As a result, many important signals, disclosures, and market events remain unnoticed.

This creates an intelligence gap during client interactions. Brokers often rely on partial information, memory, or fragmented market sources while answering client questions. Important disclosures buried inside long 10-K and 10-Q filings are difficult to review manually, and brokers may struggle to provide evidence-backed responses when clients ask for justification or supporting references. This increases both compliance risk and trust-related challenges.

To solve this problem, GlobalEdge wants to build a financial intelligence assistant that allows brokers to ask natural-language questions and receive grounded answers backed by real financial data, news articles, and regulatory filings. The system should reduce information overload, improve market coverage, and help brokers make faster and more confident advisory decisions without adding technical complexity to their workflow.

### Objective

This project proposes a proof-of-concept Retrieval-Augmented Generation (RAG) financial intelligence system that combines financial news, stock-price data, and SEC filings into a searchable intelligence layer. The system uses semantic retrieval with a local Chroma vector database to fetch relevant financial context and generate grounded answers for broker questions using large language models. Brokers can ask plain-English questions such as market sentiment analysis, company-risk queries, filing-related questions, or cross-market comparisons and receive evidence-backed responses.

To improve the quality and reliability of generated answers, the system introduces a DeepEval-based prompt optimization workflow using GEPA (Genetic-Pareto Prompt Optimization). Instead of manually refining prompts, the workflow uses benchmark financial question-answer examples and evaluation metrics such as faithfulness, groundedness, relevance, and actionability to optimize the final answering prompt.

The project also evaluates multiple RAG configurations by tuning retrieval and generation parameters such as chunk size, chunk overlap, number of retrieved chunks, temperature, top-p, and maximum token limits. Each configuration is evaluated using DeepEval metrics to identify the most effective combination for grounded financial reasoning and broker-style question answering.

The final system uses the optimized prompt together with the best-performing RAG configuration to generate accurate, grounded, and production-style financial intelligence responses for broker workflows.

### Data Description

The system uses three financial intelligence datasets collected through a custom data ingestion pipeline and stored locally for the RAG workflow.
1. Global Financial News (global_news.csv)
Contains financial news articles with titles, article content, publication dates, URLs, and source metadata. The dataset covers company announcements, market developments, macroeconomic events, sector movements, and sentiment-related signals.
2. Global Stock Prices (all_prices_clean.csv)
Contains historical stock-price data across global equity markets, crypto markets, and forex markets. Each record includes ticker symbols, company names, open/high/low/close prices, trading volume, and timestamps for market analysis and trend evaluation.
3. SEC Regulatory Filings (sec_filings.txt)
Contains regulatory filing documents including 10-K annual reports, 10-Q quarterly reports, and other SEC disclosures. These filings include financial statements, governance information, risk disclosures, operational commentary, and compliance-related information used for grounded financial reasoning.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Objective: install the compatible libraries required through Section 2.4.
# Run this once, restart the kernel as noted below, and then execute sequentially.
%pip install -qU \
    "chromadb==1.5.9" \
    "langchain-community==0.4.1" \
    "langchain-chroma==1.1.0" \
    "langchain-openai==1.2.1" \
    "langchain-text-splitters==1.1.2" \
    "pandas>=2.2.0" \
    "numpy>=1.26.0" \
    "pypdf>=5.0.0" \
    "scikit-learn>=1.4.0" \
    "python-dotenv>=1.0.1" \
    "tqdm>=4.66.0" \
    "deepeval==4.0.9"

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Importing the necessary libraries
# Objective: centralize every dependency used through Section 2.4 so that
# the notebook execution order and external requirements remain explicit.

# Standard Python libraries for file handling, text processing, JSON handling,
# randomness, timing, and basic utilities.
import os
import re
import json
import random
import time
from pathlib import Path
from collections import Counter
from hashlib import sha256
import pandas as pd
from dotenv import load_dotenv

# DeepEval libraries for prompt optimization and evaluation.
import deepeval
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from deepeval.metrics import GEval
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import GPTModel
from deepeval.optimizer.policies import TieBreaker

# LangChain libraries for document loading, text splitting, embeddings,
# vector storage, and LLM interaction.
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document

## Environment Setup and Project Initialization

In [ ]:
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
load_dotenv(env_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found. Add it to your .env file before running the notebook.")

print(f"Loaded environment variables from: {env_path.resolve()}")

# Section 1: Baseline Retrieval-Augmented Generation (RAG) Pipeline

## 1.1 Load the CSV files with `CSVLoader`

In [ ]:
# Objective: build explicit LangChain Documents without changing the raw CSVs.
# page_content is optimized for embeddings, while metadata supports filtering,
# traceability, citations, and deterministic retrieval diagnostics.

# Resolve the raw-data directory for both common execution locations:
# running from the notebooks folder or from the project root.
data_dir = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")

# Load both CSV sources into DataFrames. Missing values are replaced with
# empty strings so document text and Chroma metadata remain serializable.
stock_prices_path = data_dir / "stock_price_details.csv"
global_news_path = data_dir / "global_news.csv"
stock_df = pd.read_csv(stock_prices_path).fillna("")
news_df = pd.read_csv(global_news_path).fillna("")

# Parse the structured identifiers embedded in each human-readable price
# summary. These fields become metadata filters without modifying the CSV.
price_record_pattern = re.compile(
    r"Date is (?P<date>[^,]+), ticker is (?P<ticker>[^,]+), "
    r"name is (?P<name>[^,]+),"
)

# Convert every valid stock-price row into one LangChain Document. The full
# summary is embedded, while ticker and date support exact metadata filtering.
stock_docs = []
unparsed_price_rows = []
for row_id, row in stock_df.iterrows():
    price_summary = str(row["price_summary"]).strip()

    """     
        search() looks for a pattern match anywhere in the string, while match() only checks the start of the string.
        In this case, search() is used to find the pattern anywhere in the price_summary string
        and extract the relevant fields (date, ticker, name) from it. If a match is found, the fields are extracted 
        and used to create a Document object with the price_summary as page_content and the extracted fields as metadata. 
        If no match is found, the row_id is added to the unparsed_price_rows list for later reporting.
        
        Example of a valid price_summary string:     

        match = {
        "date": "2026-03-12",
        "ticker": "0700.HK",
        "name": "Tencent (Hong Kong)",
        } 
    
    """
    
    match = price_record_pattern.search(price_summary)

    """ 
        If no match is found, the row_id is added to the unparsed_price_rows list for later reporting, and not added to the stock_docs list. 
        This ensures that only valid and complete price records are indexed, and any issues with parsing can be addressed early on.
        This guarantees that the indexed document count remains aligned with the source dataset, and any issues with parsing can be addressed early on.
        This way all documents that are indexed have complete and accurate metadata, which is important for filtering, traceability, and provenance.
    """
    if not match:
        unparsed_price_rows.append(row_id)
        continue

    """ 
        match.groupdict() returns a dictionary containing all the named groups in the regular expression pattern.
        In this case, the named groups are "date", "ticker", and "name".

        Example of the dictionary returned by match.groupdict():
        fields = {
        "date": "2026-03-12",
        "ticker": "0700.HK",
        "name": "Tencent (Hong Kong)",
        }
    """
    fields = match.groupdict()

    """ 
        What is Document in LangChain?
        Document is a class in LangChain that represents a piece of text along with its associated metadata.
        It is used to store and manage text data in a structured way, allowing for easy retrieval and manipulation of the text and its metadata.

       """

    stock_docs.append(Document(
        page_content=price_summary,                     # the full price summary is embedded for semantic search. Chroma generates the embeddings from this field.  
        metadata={                                      # metadata fields are used for filtering, traceability, and provenance. They are not embedded.
            "dataset": "stock_price_details",           # chroma metadata filter for the source dataset:
            "source_file": stock_prices_path.name,          # chroma metadata filter for the source CSV
            "row_id": int(row_id),                          # chroma metadata filter for the source CSV row   
            "ticker": fields["ticker"].strip().upper(),     # chroma metadata filter for the stock ticker
            "date": fields["date"].strip(),                 # chroma metadata filter for the stock price date
            "name": fields["name"].strip(),                 # chroma metadata filter for the stock name
        },
    ))

# Fail early instead of silently indexing incomplete price records. This keeps
# the indexed document count aligned with the source dataset.
if unparsed_price_rows:
    raise ValueError(f"Could not parse stock rows: {unparsed_price_rows[:10]}")

# Build one news Document per CSV row. The title, date, source, description,
# and content form the semantic text; provenance fields remain in metadata.
news_docs = []
for row_id, row in news_df.iterrows():
    news_content = f"""
Title: {row['title']}
Published at: {row['published_at']}
Source: {row['source']}

Description:
{row['description']}

Content:
{row['content']}
""".strip()

    news_docs.append(Document(
        page_content=news_content,                      # the full news text is embedded for semantic search. Chroma generates the embeddings from this field.
        metadata={                                      # metadata fields are used for filtering, traceability, and provenance. They are not embedded.    
            "dataset": "global_news",                       # chroma metadata filter for the source dataset
            "source_file": global_news_path.name,           # chroma metadata filter for the source CSV
            "row_id": int(row_id),                          # chroma metadata filter for the source CSV row    
            "title": str(row["title"]),                     # chroma metadata filter for the news title
            "source": str(row["source"]),                   # chroma metadata filter for the news source
            "url": str(row["url"]),                         # chroma metadata filter for the news URL
            "published_at": str(row["published_at"]),       # chroma metadata filter for the news publication date
            "query": str(row["query"]),                     # chroma metadata filter for the news query
        },
    ))

# Combine the two CSV document collections for downstream indexing and print
# a compact ingestion check before the SEC documents are added in Section 1.2.
csv_docs = stock_docs + news_docs

print(f"Loaded {len(stock_docs)} stock price documents")
print(f"Loaded {len(news_docs)} news documents")
print(f"Loaded {len(csv_docs)} total CSV documents")
print("Sample metadata:", csv_docs[0].metadata)
print(csv_docs[0].page_content[:500])

## 1.2 Load the SEC filings text and split it into chunks


In [ ]:
# Objective: preserve page-level provenance while creating embedding-sized
# SEC chunks. The PDF remains unchanged; only in-memory metadata is enriched.
sec_pdf_path = data_dir / "sec_filings_10q.pdf"

if not sec_pdf_path.exists():
    raise FileNotFoundError("Expected sec_filings_10q.pdf in the raw data folder.")

# Load the SEC PDF into one LangChain Document per page. The PyPDFLoader
# automatically extracts text and preserves page-level metadata. 
sec_loader = PyPDFLoader(str(sec_pdf_path))

# sec_raw_docs is a list of Document objects, each representing a page in the SEC PDF. 
# Each Document object contains the text content of the page and metadata such as the page number and source file name.
sec_raw_docs = sec_loader.load()

# Fail early if the SEC PDF did not produce any page documents. This ensures that the subsequent processing steps have valid input data to work with.
if not sec_raw_docs:
    raise ValueError("The SEC PDF did not produce any page documents.")


# Enrich the SEC page-level metadata with dataset, source file, and document type.
for doc in sec_raw_docs:
    doc.metadata.update({
        "dataset": "sec_filings",
        "source_file": sec_pdf_path.name,
        "document_type": "10-Q",
    })

# Objective: create embedding-sized chunks from the SEC filings while preserving
# page-level provenance. The RecursiveCharacterTextSplitter is configured to
# produce chunks of 1000 characters with 200 characters of overlap. This ensures
# that each chunk is large enough for semantic search while maintaining context
# across chunk boundaries. The overlap helps to preserve continuity in the text,
# which is important for understanding the content in a financial document like a 10-Q filing.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

# Split the SEC documents into embedding-sized chunks.
sec_docs = text_splitter.split_documents(sec_raw_docs)

# Add a unique chunk ID to each SEC chunk for tracking and retrieval.
for chunk_id, doc in enumerate(sec_docs):
    doc.metadata["chunk_id"] = chunk_id

# Fail early if the SEC chunking produced empty content. This ensures that the subsequent processing steps have valid input data to work with.
if not sec_docs or any(not doc.page_content.strip() for doc in sec_docs):
    raise ValueError("SEC chunking produced empty content.")

# Combine the CSV documents with the SEC chunks for final indexing and print a compact ingestion check. 
# This ensures that all documents are ready for vector indexing and semantic search.
all_docs = csv_docs + sec_docs

print(f"Loaded {len(sec_raw_docs)} SEC filing source documents")
print(f"Created {len(sec_docs)} SEC filing chunks")
print(f"Prepared {len(all_docs)} total documents for vector indexing")
print(sec_docs[0].page_content[:500])

## 1.3 Build a local Chroma vector store

In [ ]:
# Objective: rebuild a deterministic Version 2 collection on every run.
# This prevents duplicate documents from changing top-k results after a cell rerun.
#
# Model rationale: text-embedding-3-small was selected as an efficient and
# cost-effective embedding model with strong semantic retrieval capabilities.
# It converts news articles, stock-price records, and SEC filing chunks into
# comparable vectors in one Chroma collection. Its quality is sufficient for
# this proof-of-concept corpus while keeping repeated indexing and configuration
# experiments fast, affordable, and reproducible. Although
# text-embedding-3-large may improve retrieval in more demanding production
# workloads, the small model provides the appropriate balance of retrieval
# performance, speed, storage efficiency, and cost for this project.
chroma_dir = Path("../chroma_db") if Path("../data/raw").exists() else Path("chroma_db")
collection_name = "investment_rag_v2"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a new Chroma client and delete the existing collection if it exists
chroma_client = chromadb.PersistentClient(path=str(chroma_dir))
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    # A missing collection is expected on the first run.
    pass

# Define a function to generate deterministic document IDs based on the document's metadata and content. 
# This ensures that each document has a unique identifier that can be used for indexing and retrieval.
def deterministic_document_id(doc):
    identity = json.dumps(doc.metadata, sort_keys=True) + doc.page_content
    return sha256(identity.encode("utf-8")).hexdigest()

# Generate deterministic document IDs for all documents and check for duplicates before indexing. 
# This ensures that each document can be uniquely identified and retrieved later.
document_ids = [deterministic_document_id(doc) for doc in all_docs]
if len(document_ids) != len(set(document_ids)):
    raise ValueError("Duplicate document identities were detected before indexing.")

# Create a new Chroma collection with the specified name, embedding function, and client.
vector_store = Chroma(
    client=chroma_client,
    collection_name=collection_name,
    embedding_function=embeddings,
)

# Add all documents to the Chroma collection with their corresponding deterministic IDs. 
# This ensures that each document is indexed and can be retrieved later using its unique identifier.
vector_store.add_documents(all_docs, ids=document_ids)

# Create a retriever from the vector store with similarity search and a top-k of 5.
# This allows for efficient retrieval of the most relevant documents based on their embeddings when a query is made.
# search_type="similarity" specifies that the retriever will use cosine similarity to find the most relevant documents based on their embeddings.
# search_kwargs={"k": 5} specifies that the retriever will return the top 5 most similar documents for a given query.
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

dataset_counts = Counter(doc.metadata["dataset"] for doc in all_docs)
print(f"Created Chroma collection: {collection_name}")
print(f"Persisted vector store at: {chroma_dir.resolve()}")
print(f"Indexed {vector_store._collection.count()} documents")
print("Documents by dataset:", dict(dataset_counts))

## 1.4 Retrieve Context and Generate Answers using the Baseline RAG Pipeline

In [ ]:
# Objective: make retrieval observable and source-aware while preserving a
# transparent similarity-search baseline. Routing uses only the user question.
FINANCIAL_SAFETY_POLICY = """
Do not provide personalized buy, sell, or hold recommendations. Present the
available evidence, risks, signals, and data limitations so the broker or client
can make an informed decision.
""".strip()

baseline_system_prompt = f"""
You are a grounded investment intelligence assistant for brokerage analysts.
Use only the retrieved context to answer the user's question.
If the context is insufficient, say what is missing instead of guessing.
Keep the answer concise, evidence-backed, and suitable for a broker speaking with a client.
{FINANCIAL_SAFETY_POLICY}
""".strip()

# Define a function to format source metadata for display. 
def format_source_metadata(metadata):
    source_file = metadata.get("source_file") or metadata.get("source") or metadata.get("dataset", "unknown")
    title = metadata.get("title")
    published_at = metadata.get("published_at")
    page = metadata.get("page")
    row = metadata.get("row_id", metadata.get("row"))

    details = [str(source_file)]
    if title:
        details.append(str(title))
    if published_at:
        details.append(str(published_at))
    if page is not None:
        details.append(f"page {page}")
    if row is not None:
        details.append(f"row {row}")

    return " | ".join(details)

# Define a function to format the retrieved context for display.
def format_retrieved_context(docs):
    formatted_sources = []

    for idx, doc in enumerate(docs, start=1):
        metadata = doc.metadata or {}
        source_label = format_source_metadata(metadata)
        content = re.sub(r"\s+", " ", doc.page_content).strip()
        formatted_sources.append(f"[Source {idx}] {content}\nMetadata: {source_label}")

    source_text = "\n\n".join(formatted_sources)
    return f"Corpus snapshot date: {CORPUS_AS_OF_DATE}\n\n{source_text}"

# Define sets of terms to help route questions to the appropriate datasets. 
PRICE_TERMS = {
    "price", "close", "closing", "open", "high", "low",
    "volume", "trading volume", "exchange rate", "stock",
    "price trend", "performing",
}
SEC_TERMS = {
    "10-k", "10-q", "sec", "filing", "auditor", "audit",
    "internal control", "material weakness", "financial statements",
    "financials", "revenue", "net income", "risk", "risks",
}
NEWS_TERMS = {
    "news", "recent reports", "market chatter", "announced",
    "outlook", "blockade", "recently", "geopolitical",
    "exchange and ticker", "sentiment", "buzz", "stepping down",
}

# Build aliases from stock metadata and add only the common-name exceptions
# that cannot be derived reliably from the source labels.
COMPANY_TICKER_ALIASES = {}
for doc in stock_docs:
    company_name = str(doc.metadata["name"]).strip().lower()
    base_name = re.sub(r"\s*\([^)]*\)\s*$", "", company_name).strip()
    COMPANY_TICKER_ALIASES[company_name] = doc.metadata["ticker"]
    COMPANY_TICKER_ALIASES[base_name] = doc.metadata["ticker"]

COMPANY_TICKER_ALIASES.update({
    "apple": "AAPL",
    "amazon": "AMZN",
    "alphabet": "GOOGL",
    "google": "GOOGL",
    "tesla": "TSLA",
    "tencent": "0700.HK",
    "bitcoin": "BTC-USD",
    "usd/jpy": "USDJPY=X",
    "usdjpy": "USDJPY=X",
})

# Build a sorted list of known tickers from the stock documents for efficient extraction.
KNOWN_TICKERS = sorted({doc.metadata["ticker"] for doc in stock_docs}, key=len, reverse=True)

# Derive the corpus cutoff from the available news and price records instead
# of hard-coding a date that could become stale when the raw files change.
news_dates = pd.to_datetime(news_df["published_at"], errors="coerce", utc=True)
price_dates = pd.to_datetime(
    [doc.metadata["date"] for doc in stock_docs], errors="coerce", utc=True
)
available_dates = pd.concat([
    pd.Series(news_dates).dropna(),
    pd.Series(price_dates).dropna(),
])
if available_dates.empty:
    raise ValueError("Could not derive the corpus snapshot date.")
CORPUS_AS_OF_DATE = available_dates.max().date().isoformat()

# Define a function to route questions to the appropriate datasets based on keywords in the question.
def route_question(question):
    text = question.lower()
    constraints = extract_question_constraints(question)
    datasets = []
    if any(term in text for term in PRICE_TERMS):
        datasets.append("stock_price_details")
    if any(term in text for term in SEC_TERMS):
        datasets.append("sec_filings")
    if any(term in text for term in NEWS_TERMS):
        datasets.append("global_news")
    if len(constraints["tickers"]) > 1 and any(
        term in text for term in {"invest", "best bet"}
    ):
        datasets = ["global_news", "stock_price_details", "sec_filings"]
    return datasets or ["global_news", "stock_price_details", "sec_filings"]

# Define a function to extract constraints from the question, such as ticker symbols and dates.
def extract_question_constraints(question):
    text = question.lower()
    tickers = {
        ticker for ticker in KNOWN_TICKERS if ticker.lower() in text
    }
    tickers.update({
        ticker
        for alias, ticker in COMPANY_TICKER_ALIASES.items()
        if alias and alias in text
    })
    dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", question)
    return {"tickers": sorted(tickers), "dates": dates}

# Define a function to create a Chroma filter based on the dataset, ticker, and date constraints.
def chroma_filter(dataset, ticker=None, date=None):
    conditions = [{"dataset": {"$eq": dataset}}]
    if dataset == "stock_price_details" and ticker:
        conditions.append({"ticker": {"$eq": ticker}})
    if dataset == "stock_price_details" and date:
        conditions.append({"date": {"$eq": date}})
    return conditions[0] if len(conditions) == 1 else {"$and": conditions}

# Define a function to retrieve context from the vector store based on the question, constraints, and search type.
def retrieve_context(question, k=5, search_type="similarity", use_routing=True):
    datasets = route_question(question) if use_routing else ["global_news", "stock_price_details", "sec_filings"]
    constraints = extract_question_constraints(question)
    k_per_dataset = k if len(datasets) == 1 else max(2, k // len(datasets)) # Distribute the top-k results across datasets
    retrieved_docs = []
    searches = []

    # Iterate through each dataset and perform a similarity search or max marginal relevance search based on the specified search type. 
    for dataset in datasets:
        requested_date = constraints["dates"][0] if len(constraints["dates"]) == 1 else None
        ticker_targets = (
            constraints["tickers"]
            if dataset == "stock_price_details" and constraints["tickers"]
            else [None]
        )
        k_per_target = max(1, k_per_dataset // len(ticker_targets))

        for ticker in ticker_targets:
            where = chroma_filter(dataset, ticker=ticker, date=requested_date)
            if search_type == "mmr":
                docs = vector_store.max_marginal_relevance_search(
                    question, k=k_per_target,
                    fetch_k=max(20, k_per_target * 4), filter=where
                )
            else:
                docs = vector_store.similarity_search(
                    question, k=k_per_target, filter=where
                )

            fallback_used = False
            if not docs and dataset == "stock_price_details" and requested_date:
                fallback_used = True
                where = chroma_filter(dataset, ticker=ticker)
                docs = vector_store.similarity_search(
                    question, k=k_per_target, filter=where
                )

            retrieved_docs.extend(docs)
            searches.append({
                "dataset": dataset,
                "ticker": ticker,
                "filter": where,
                "result_count": len(docs),
                "fallback_used": fallback_used,
            })

    unique_docs = []
    seen_ids = set()
    for doc in retrieved_docs:
        doc_id = deterministic_document_id(doc)
        if doc_id not in seen_ids:
            seen_ids.add(doc_id)
            unique_docs.append(doc)

    diagnostics = {
        "datasets_requested": datasets,
        "constraints": constraints,
        "search_type": search_type,
        "searches": searches,
        "retrieved_count": len(unique_docs),
        "retrieved_datasets": sorted({doc.metadata["dataset"] for doc in unique_docs}),
    }
    return {"documents": unique_docs, "diagnostics": diagnostics}


# Model rationale: GPT-4.1 mini was selected as a fast and cost-efficient
# generation model with strong instruction-following capabilities. These
# characteristics help the RAG workflow remain grounded in retrieved evidence,
# cite sources, recognize missing information, and support repeated prompt
# optimization and configuration-evaluation calls. Although newer models may
# provide higher capability for production workloads, GPT-4.1 mini offers an
# appropriate balance of quality, latency, cost, and experimental
# reproducibility for this proof of concept. Temperature zero further reduces
# generation variability during controlled comparisons.
baseline_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Define a function to answer a question using the baseline retrieval-augmented generation (RAG) approach.
def baseline_rag_answer(question, k=5):
    retrieval_result = retrieve_context(question, k=k)
    retrieved_docs = retrieval_result["documents"]
    context = format_retrieved_context(retrieved_docs)

    user_prompt = f"""
Question:
{question}

Retrieved context:
{context}

Answer with a direct response followed by a short Sources section naming the most relevant source numbers.
""".strip()

    response = baseline_llm.invoke([
        SystemMessage(content=baseline_system_prompt),
        HumanMessage(content=user_prompt),
    ])

    return {
        "question": question,
        "answer": response.content,
        "context": context,
        "source_documents": retrieved_docs,
        "retrieval_diagnostics": retrieval_result["diagnostics"],
    }


sample_question = "What are the main risks or signals a broker should discuss with a client considering Apple?"
baseline_sample = baseline_rag_answer(sample_question)

print("Question:")
print(baseline_sample["question"])

print("\nAnswer:")
print(baseline_sample["answer"])

print("\nBEGIN Context:")
print(baseline_sample["context"])
print("\nEND Context:")

print("\nRetrieved sources:")
for idx, doc in enumerate(baseline_sample["source_documents"], start=1):
    print(f"[{idx}] {format_source_metadata(doc.metadata or {})}")

print("\nRetrieval diagnostics:")    
print(baseline_sample["retrieval_diagnostics"])



## 1.5 Load the gold benchmark dataset and evaluate the baseline prompt


In [ ]:
# Objective: create a leakage-resistant benchmark workflow. GEPA optimization
# and final prompt evaluation use separate, source-balanced subsets. Retrieval
# is executed once and cached so both prompts receive identical evidence.
gold_dataset_path = Path("../data/eval/golden_benchmark_dataset.csv")
if not gold_dataset_path.exists():
    gold_dataset_path = Path("data/eval/golden_benchmark_dataset.csv")

gold_df = pd.read_csv(gold_dataset_path)

# Validate that the gold benchmark dataset contains all required columns. 
# If any required columns are missing, raise a ValueError with a message indicating which columns are missing. 
# This ensures that the dataset is complete and suitable for evaluation.
required_columns = {"question", "response", "source_hint", "supporting_sources", "context"}
missing_columns = required_columns.difference(gold_df.columns)
if missing_columns:
    raise ValueError(f"Gold benchmark dataset is missing columns: {sorted(missing_columns)}")

# Drop any rows with missing values in the required columns and reset the index. 
# This ensures that the benchmark dataset is clean and ready for evaluation.
gold_df = gold_df.dropna(subset=["question", "response"]).reset_index(drop=True)

# gold_df.index is a RangeIndex starting from 0, so adding 1 to it will create a new column "case_id" that starts from 1.
# This is useful for tracking and referencing individual cases in the benchmark dataset.
gold_df["case_id"] = gold_df.index + 1

# Map the supporting source labels in the gold benchmark dataset to their corresponding expected dataset names.
SOURCE_DATASET_MAP = {
    "global_news.csv": "global_news",
    "all_prices_clean.csv": "stock_price_details",
    "stock_price_details.csv": "stock_price_details",
    "sec_filings.txt": "sec_filings",
    "sec_filings_10q.pdf": "sec_filings",
}
gold_df["expected_dataset"] = gold_df["supporting_sources"].map(SOURCE_DATASET_MAP)
if gold_df["expected_dataset"].isna().any():
    unknown_sources = sorted(gold_df.loc[gold_df["expected_dataset"].isna(), "supporting_sources"].unique())
    raise ValueError(f"Unknown supporting source labels: {unknown_sources}")


# Reserve approximately 40% of the cases from each expected source for holdout evaluation.
# The remaining cases are used for GEPA prompt optimization.
# Keeping optimization and evaluation separate prevents overfitting and data leakage:
# - Overfitting: optimizing the prompt for a specific set of cases instead of generalizing.
# - Data leakage: allowing evaluation cases to influence prompt optimization, leading to overly optimistic results.
# Grouping by `expected_dataset` preserves representation from each source, while
# `random_state=42` makes the split reproducible.


evaluation_df = (
    gold_df.groupby("expected_dataset", group_keys=False)
    .sample(frac=0.4, random_state=42)
    .sort_values("case_id")
    .reset_index(drop=True)
)
optimization_df = (
    gold_df[~gold_df["case_id"].isin(evaluation_df["case_id"])]
    .sort_values("case_id")
    .reset_index(drop=True)
)

# Objective: cache the retrieval results for every gold benchmark question. This
# ensures that both the GEPA prompt optimization and final evaluation use identical
# evidence. The retrieval diagnostics are also cached for later analysis.
benchmark_contexts = {}
for _, row in gold_df.iterrows():
    retrieval_result = retrieve_context(row["question"], k=5)
    docs = retrieval_result["documents"]
    benchmark_contexts[row["question"]] = {
        "documents": docs,
        "formatted_context": format_retrieved_context(docs),
        "retrieval_context": [doc.page_content for doc in docs],
        "retrieved_sources": [format_source_metadata(doc.metadata or {}) for doc in docs],
        "diagnostics": retrieval_result["diagnostics"],
    }

# Define a function to evaluate the quality of the retrieval results against the expected dataset, ticker, and date constraints.
def retrieval_quality(row, cached_context):
    docs = cached_context["documents"]
    retrieved_datasets = sorted({doc.metadata["dataset"] for doc in docs})
    expected_dataset = row["expected_dataset"]
    constraints = extract_question_constraints(row["question"])
    source_match = expected_dataset in retrieved_datasets
    ticker_match = (
        None
        if expected_dataset != "stock_price_details" or not constraints["tickers"]
        else all(
            any(doc.metadata.get("ticker") == ticker for doc in docs)
            for ticker in constraints["tickers"]
        )
    )
    requested_dates = constraints["dates"]
    date_match = (
        None if expected_dataset != "stock_price_details" or not requested_dates else
        all(any(doc.metadata.get("date") == date for doc in docs) for date in requested_dates)
    )
    if not source_match:
        status = "UNSUPPORTED"
    elif ticker_match is False or date_match is False:
        status = "PARTIAL"
    else:
        status = "SUPPORTED"
    return {
        "expected_dataset": expected_dataset,
        "retrieved_datasets": retrieved_datasets,
        "source_match": source_match,
        "ticker_match": ticker_match,
        "date_match": date_match,
        "unique_sources": len(set(cached_context["retrieved_sources"])),
        "coverage_status": status,
    }

# Objective: generate baseline outputs for every holdout evaluation case. The
# baseline RAG approach uses the same retrieval evidence as the GEPA prompt optimization.
baseline_eval_df = evaluation_df.copy()


baseline_rows = []
baseline_test_cases = []

# Iterate through each row in the baseline evaluation DataFrame and generate a response using the baseline RAG approach.
for _, row in baseline_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["response"]
    cached = benchmark_contexts[question]
    retrieval_checks = retrieval_quality(row, cached)

    user_prompt = f"""
Question:
{question}

Retrieved context:
{cached['formatted_context']}

Answer with a direct response followed by a short Sources section naming the most relevant source numbers.
""".strip()
    
    # Invoke the baseline LLM with the system prompt and user prompt to generate a response.
    response = baseline_llm.invoke([
        SystemMessage(content=baseline_system_prompt),
        HumanMessage(content=user_prompt),
    ])

    # Store the results in a dictionary and append it to the baseline_rows list for later analysis.
    baseline_rows.append({
        "case_id": int(row["case_id"]),
        "question": question,
        "expected_output": expected_answer,
        "actual_output": response.content,
        "retrieved_context": cached["formatted_context"],
        "retrieved_sources": cached["retrieved_sources"],
        "source_hint": row["source_hint"],
        "supporting_sources": row["supporting_sources"],
        **retrieval_checks,
    })

    # Create an LLMTestCase object for the current question and append it to the baseline_test_cases list for later evaluation.
    baseline_test_cases.append(
        LLMTestCase(
            input=question,
            actual_output=response.content,
            expected_output=expected_answer,
            retrieval_context=cached["retrieval_context"],
        )
    )

# Create a DataFrame from the baseline_rows list for easier analysis and display.
baseline_results_df = pd.DataFrame(baseline_rows)

print(f"Loaded {len(gold_df)} gold benchmark examples from {gold_dataset_path.resolve()}")
print(f"Optimization cases: {len(optimization_df)} | Holdout evaluation cases: {len(evaluation_df)}")
print(f"Generated baseline outputs for {len(baseline_results_df)} holdout examples")
display(baseline_results_df[["case_id", "expected_dataset", "retrieved_datasets", "coverage_status", "source_match"]])

## 1.6 Define the evaluation metrics

In [ ]:
# Objective: define separate retrieval and prompt-quality metrics with an
# explicit success threshold. Deterministic retrieval checks remain in 1.5.
EVALUATION_THRESHOLD = 0.5

# Evaluator-model rationale: GPT-4.1 mini is used explicitly as the DeepEval
# judge because its strong instruction-following capability is suitable for
# applying detailed metric criteria consistently. Its low latency and cost also
# make repeated evaluation across prompts, cases, and RAG configurations practical
# for this proof of concept. Reusing the same model family supports a simple and
# reproducible experiment, although a production evaluation should validate these
# LLM-as-a-judge results with a stronger independent model and human review to
# detect correlated preferences or scoring bias.
EVALUATOR_MODEL_NAME = "gpt-4.1-mini"
evaluation_model = GPTModel(model=EVALUATOR_MODEL_NAME)
print(f"DeepEval evaluator model: {EVALUATOR_MODEL_NAME}")

# Define a GEval metric for answer correctness, which evaluates whether the actual output 
# correctly answers the input question and matches the expected output from the gold benchmark dataset. 
# The evaluation parameters include the input question, actual output, and expected output. 
# The threshold for passing this metric is set to 0.5, meaning that at least 50% of the cases must meet 
# the criteria for the metric to be considered successful.
correctness_metric = GEval(
    name="Answer Correctness",
    threshold=EVALUATION_THRESHOLD,
    model=evaluation_model,
    criteria=(
        "Evaluate whether the actual output correctly answers the input question "
        "and matches the expected output from the gold benchmark dataset."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    async_mode=False,
)

# Define a GEval metric for faithfulness, which evaluates whether the actual output is fully supported by the retrieved context.
# This metric penalizes unsupported claims, hallucinations, or facts not present in the context.    
faithfulness_metric = GEval(
    name="Faithfulness / Groundedness",
    threshold=EVALUATION_THRESHOLD,
    model=evaluation_model,
    criteria=(
        "Evaluate whether the actual output is fully supported by the retrieved context. "
        "Penalize unsupported claims, hallucinations, or facts not present in the context."
    ),
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    async_mode=False,
)

# Define a GEval metric for context relevance, which evaluates whether the retrieved context 
# is relevant and sufficient to answer the input question.
# This is a retriever metric rather than a prompt quality metric, so it is not used for GEPA optimization.
context_relevance_metric = GEval(
    name="Context Relevance",
    threshold=EVALUATION_THRESHOLD,
    model=evaluation_model,
    criteria=(
        "Evaluate whether the retrieved context is relevant and sufficient to answer "
        "the input question."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    async_mode=False,
)

# Define a GEval metric for answer relevance, which evaluates whether the actual output 
# directly answers the input question without unnecessary or unrelated information.
answer_relevance_metric = GEval(
    name="Answer Relevance",
    threshold=EVALUATION_THRESHOLD,
    model=evaluation_model,
    criteria=(
        "Evaluate whether the actual output directly answers the input question "
        "without unnecessary or unrelated information."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
    ],
    async_mode=False,
)

# Define a GEval metric for broker actionability, which evaluates whether 
# the actual output is useful for a broker speaking with a client.
broker_actionability_metric = GEval(
    name="Broker Actionability",
    threshold=EVALUATION_THRESHOLD,
    model=evaluation_model,
    criteria=(
        "Evaluate whether the answer is useful for a broker speaking with a client. "
        "The answer should highlight relevant risks, signals, evidence, and practical implications."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    async_mode=False,
)

# Retrieval quality is evaluated once for each frozen question-context pair.
# It is reported independently because the prompt does not control retrieval.
retrieval_metrics = [
    context_relevance_metric,
]

# These metrics can change when the generation prompt changes. They are used
# to compare the baseline and optimized prompts on identical evidence.
prompt_evaluation_metrics = [
    correctness_metric,
    faithfulness_metric,
    answer_relevance_metric,
    broker_actionability_metric,
]

# This combined list describes the complete RAG evaluation suite. It is not
# used directly for prompt comparison because it mixes retrieval and generation.
rag_evaluation_metrics = retrieval_metrics + prompt_evaluation_metrics

# The five official project questions do not include human-reviewed reference
# answers. Evaluate them with the four metrics that do not require
# EXPECTED_OUTPUT; do not claim an Answer Correctness score for those cases.
official_test_evaluation_metrics = [
    context_relevance_metric,
    faithfulness_metric,
    answer_relevance_metric,
    broker_actionability_metric,
]

# GEPA optimizes only metrics that the generation prompt can influence.
prompt_optimization_metrics = prompt_evaluation_metrics.copy()

## 1.7 Evaluate the baseline RAG on the golden examples dataset

In [ ]:
# Objective: evaluate retrieval once per frozen context, then score the
# baseline answer only with metrics that the generation prompt can influence.
def measure_metric(metric, test_case):
    try:
        metric.measure(test_case, _show_indicator=False)
        return {
            "score": metric.score,
            "success": metric.is_successful() if hasattr(metric, "is_successful") else None,
            "reason": metric.reason,
            "error": None,
        }
    except Exception as exc:
        return {"score": None, "success": False, "reason": None, "error": str(exc)}


# Define a function to summarize the evaluation metrics by calculating the average score, number of evaluated cases, number of successful cases, and number of failed or error cases for each metric.
def summarize_metrics(evaluation_df):
    return (
        evaluation_df.groupby("metric", dropna=False)
        .agg(
            average_score=("score", "mean"),
            evaluated_cases=("score", "count"),
            successful_cases=("success", "sum"),
            failed_or_error_cases=("error", lambda values: values.notna().sum()),
        )
        .reset_index()
        .sort_values("average_score", ascending=False)
    )


# Context Relevance depends only on the question and frozen retrieval context.
# Evaluate it once instead of repeating it for both generation prompts.
retrieval_evaluation_rows = []
for baseline_row, test_case in zip(baseline_rows, baseline_test_cases):
    for metric in retrieval_metrics:
        retrieval_evaluation_rows.append({
            "case_id": baseline_row["case_id"],
            "question": test_case.input,
            "metric": metric.name,
            "coverage_status": baseline_row["coverage_status"],
            "source_match": baseline_row["source_match"],
            **measure_metric(metric, test_case),
        })

retrieval_evaluation_df = pd.DataFrame(retrieval_evaluation_rows)
retrieval_metric_summary_df = summarize_metrics(retrieval_evaluation_df)


# Score baseline answers with prompt-controlled generation metrics.
baseline_evaluation_rows = []
for case_number, (baseline_row, test_case) in enumerate(
    zip(baseline_rows, baseline_test_cases), start=1
):
    case_id = baseline_row["case_id"]
    print(f"Evaluating baseline case {case_id} ({case_number}/{len(baseline_test_cases)})")

    for metric in prompt_evaluation_metrics:
        baseline_evaluation_rows.append({
            "case_id": case_id,
            "question": test_case.input,
            "metric": metric.name,
            "actual_output": test_case.actual_output,
            "expected_output": test_case.expected_output,
            "coverage_status": baseline_row["coverage_status"],
            "source_match": baseline_row["source_match"],
            **measure_metric(metric, test_case),
        })

baseline_evaluation_df = pd.DataFrame(baseline_evaluation_rows)
baseline_metric_summary_df = summarize_metrics(baseline_evaluation_df)
baseline_scores_pivot_df = baseline_evaluation_df.pivot(
    index="case_id", columns="metric", values="score"
).reset_index()

print("Frozen-context retrieval evaluation")
display(retrieval_metric_summary_df)
print("Baseline prompt evaluation")
display(baseline_metric_summary_df)
display(baseline_scores_pivot_df)
print("Deterministic retrieval coverage")
display(baseline_results_df["coverage_status"].value_counts().rename_axis("status").reset_index(name="cases"))

# Section 2: DeepEval Prompt Optimization Layer

## 2.1 Define the prompt template and model callback


In [ ]:
# Objective: optimize only the prompt. Every candidate receives a frozen
# cached context, so retrieval cannot confound the GEPA comparison.
optimized_prompt_seed = Prompt(
    text_template="""
You are a grounded investment intelligence assistant for brokerage analysts.

Use only the retrieved context to answer the question. Do not invent facts.
If the context does not contain enough evidence, clearly state what is missing.
Write a concise client-ready answer that explains the relevant financial signal,
risk, or implication. Reference the most relevant source numbers when useful.

Question:
{question}

Retrieved context:
{context}

Answer:
""".strip()
)
seed_prompt_text = optimized_prompt_seed.text_template

prompt_optimization_goldens = [
    Golden(
        input=row["question"],
        expected_output=row["response"],
        additional_metadata={
            "source_hint": row["source_hint"],
            "supporting_sources": row["supporting_sources"],
            "case_id": int(row["case_id"]),
        },
    )
    for _, row in optimization_df.iterrows()
]


def optimized_prompt_model_callback(prompt, golden):
    question = golden.input
    cached = benchmark_contexts[question]
    golden.retrieval_context = cached["retrieval_context"]

    rendered_prompt = prompt.interpolate(
        question=question,
        context=cached["formatted_context"],
    )

    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    response = llm.invoke([HumanMessage(content=rendered_prompt)])

    return response.content


print(f"Prepared {len(prompt_optimization_goldens)} goldens for prompt optimization")
print("Seed prompt preview:")
print(optimized_prompt_seed.text_template[:700])

## 2.2 Optimize the prompt with GEPA

In [ ]:
# Objective: run a low-cost, reproducible GEPA optimization using only the
# optimization split. The holdout cases remain unseen until Section 2.3.
from deepeval.optimizer.configs import AsyncConfig, DisplayConfig
from deepeval.optimizer.scorer.scorer import Scorer
from deepeval.optimizer.rewriter.rewriter import Rewriter

print(f"DeepEval version: {getattr(deepeval, '__version__', 'unknown')}")

# Workaround for a DeepEval GEPA token-accounting bug in this installed version.
# GEPA calls _accrue_tokens on helper classes that do not define it.
def _optimizer_helper_accrue_tokens(self, input_tokens=0, output_tokens=0):
    self.input_tokens = getattr(self, "input_tokens", 0) + (input_tokens or 0)
    self.output_tokens = getattr(self, "output_tokens", 0) + (output_tokens or 0)


for helper_cls in (Scorer, Rewriter):
    if not hasattr(helper_cls, "_accrue_tokens"):
        helper_cls._accrue_tokens = _optimizer_helper_accrue_tokens

gepa_algorithm = GEPA(
    iterations=3,
    minibatch_size=8,
    pareto_size=5,
    patience=2,
    random_seed=42,
    tie_breaker=TieBreaker.PREFER_CHILD,
    reflection_model="gpt-4.1-mini",
    mutation_model="gpt-4.1-mini",
)

prompt_optimizer = PromptOptimizer(
    model_callback=optimized_prompt_model_callback,
    metrics=prompt_optimization_metrics,
    optimizer_model="gpt-4.1-mini",
    algorithm=gepa_algorithm,
    async_config=AsyncConfig(run_async=False, max_concurrent=1, throttle_value=0.5),
    display_config=DisplayConfig(show_indicator=False),
)

optimized_prompt = prompt_optimizer.optimize(
    prompt=optimized_prompt_seed,
    goldens=prompt_optimization_goldens,
)

optimization_report = prompt_optimizer.optimization_report
optimized_prompt_text = optimized_prompt.text_template

print("Optimized prompt:")
print(optimized_prompt_text)


## 2.3 Evaluate the optimized prompt on the benchmark dataset

In [ ]:
# Objective: evaluate the optimized prompt only on the holdout split. The
# cached baseline contexts are reused exactly, isolating the prompt as the
# single experimental variable.
optimized_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Generate answers with the optimized prompt using the same benchmark and
# retrieval settings used for the baseline evaluation.
optimized_rows = []
optimized_test_cases = []

for _, row in evaluation_df.iterrows():
    question = row["question"]
    expected_answer = row["response"]
    cached = benchmark_contexts[question]
    retrieval_checks = retrieval_quality(row, cached)

    rendered_prompt = optimized_prompt.interpolate(
        question=question,
        context=cached["formatted_context"],
    )

    response = optimized_llm.invoke([
        HumanMessage(content=rendered_prompt)
    ])
    actual_answer = response.content

    optimized_rows.append({
        "case_id": int(row["case_id"]),
        "question": question,
        "expected_output": expected_answer,
        "actual_output": actual_answer,
        "retrieved_context": cached["formatted_context"],
        "retrieved_sources": cached["retrieved_sources"],
        "source_hint": row["source_hint"],
        "supporting_sources": row["supporting_sources"],
        **retrieval_checks,
    })

    optimized_test_cases.append(
        LLMTestCase(
            input=question,
            actual_output=actual_answer,
            expected_output=expected_answer,
            retrieval_context=cached["retrieval_context"],
        )
    )

optimized_results_df = pd.DataFrame(optimized_rows)

print(f"Generated optimized-prompt outputs for {len(optimized_results_df)} examples")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 300)
display(
    optimized_results_df[
        ["question", "expected_output", "actual_output", "supporting_sources"]
    ]
)


# Evaluate the optimized answers with the same metrics used for the baseline.
optimized_evaluation_rows = []

for case_number, (optimized_row, test_case) in enumerate(
    zip(optimized_rows, optimized_test_cases), start=1
):
    case_id = optimized_row["case_id"]
    print(
        f"Evaluating optimized-prompt case {case_id} "
        f"({case_number}/{len(optimized_test_cases)})"
    )

    for metric in prompt_evaluation_metrics:
        optimized_evaluation_rows.append({
            "case_id": case_id,
            "question": test_case.input,
            "metric": metric.name,
            "actual_output": test_case.actual_output,
            "expected_output": test_case.expected_output,
            "coverage_status": optimized_row["coverage_status"],
            "source_match": optimized_row["source_match"],
            **measure_metric(metric, test_case),
        })

optimized_evaluation_df = pd.DataFrame(optimized_evaluation_rows)

optimized_metric_summary_df = summarize_metrics(optimized_evaluation_df)

print("Optimized prompt evaluation summary")
display(optimized_metric_summary_df)

# Present one row per benchmark case, with each metric as a column.
# The full long-form results remain available in optimized_evaluation_df.
optimized_scores_pivot_df = optimized_evaluation_df.pivot(
    index="case_id",
    columns="metric",
    values="score",
).reset_index()

print("Optimized prompt scores by benchmark case")
display(optimized_scores_pivot_df)


## 2.4 Compare optimized prompt with the baseline prompt


In [ ]:
# Objective: compare prompts at metric, case, and overall levels, then apply
# quality gates so a small average gain cannot hide a critical regression.
baseline_comparison_df = baseline_metric_summary_df.rename(
    columns={
        "average_score": "baseline_average_score",
        "evaluated_cases": "baseline_evaluated_cases",
        "successful_cases": "baseline_successful_cases",
        "failed_or_error_cases": "baseline_error_cases",
    }
)

optimized_comparison_df = optimized_metric_summary_df.rename(
    columns={
        "average_score": "optimized_average_score",
        "evaluated_cases": "optimized_evaluated_cases",
        "successful_cases": "optimized_successful_cases",
        "failed_or_error_cases": "optimized_error_cases",
    }
)

prompt_comparison_df = (
    baseline_comparison_df
    .merge(optimized_comparison_df, on="metric", how="outer")
)

prompt_comparison_df["score_delta"] = (
    prompt_comparison_df["optimized_average_score"]
    - prompt_comparison_df["baseline_average_score"]
)

nonzero_baseline_scores = prompt_comparison_df[
    "baseline_average_score"
].replace(0, pd.NA)

prompt_comparison_df["improvement_pct"] = (
    prompt_comparison_df["score_delta"]
    .div(nonzero_baseline_scores)
    .mul(100)
)


def select_winner(row, tolerance=1e-9):
    baseline_score = row["baseline_average_score"]
    optimized_score = row["optimized_average_score"]

    if pd.isna(baseline_score) or pd.isna(optimized_score):
        return "Unavailable"
    if abs(optimized_score - baseline_score) <= tolerance:
        return "Tie"
    if optimized_score > baseline_score:
        return "Optimized"
    return "Baseline"


prompt_comparison_df["winner"] = prompt_comparison_df.apply(
    select_winner,
    axis=1,
)

prompt_comparison_df = prompt_comparison_df.sort_values(
    "score_delta",
    ascending=False,
).reset_index(drop=True)

print("Baseline vs. optimized prompt — metric summary")
display(
    prompt_comparison_df[
        [
            "metric",
            "baseline_average_score",
            "optimized_average_score",
            "score_delta",
            "improvement_pct",
            "winner",
            "baseline_error_cases",
            "optimized_error_cases",
        ]
    ].round(4)
)


# Compare both prompts for each benchmark case and metric.
baseline_case_scores_df = baseline_evaluation_df[
    ["case_id", "question", "metric", "score"]
].rename(columns={"score": "baseline_score"})

optimized_case_scores_df = optimized_evaluation_df[
    ["case_id", "question", "metric", "score"]
].rename(columns={"score": "optimized_score"})

case_level_comparison_df = baseline_case_scores_df.merge(
    optimized_case_scores_df,
    on=["case_id", "question", "metric"],
    how="inner",
)

case_level_comparison_df["score_delta"] = (
    case_level_comparison_df["optimized_score"]
    - case_level_comparison_df["baseline_score"]
)

case_level_comparison_df["outcome"] = case_level_comparison_df.apply(
    lambda row: select_winner(
        pd.Series({
            "baseline_average_score": row["baseline_score"],
            "optimized_average_score": row["optimized_score"],
        })
    ),
    axis=1,
)

case_level_outcomes_df = (
    case_level_comparison_df
    .groupby(["metric", "outcome"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("Per-case comparison counts")
display(case_level_outcomes_df)


# Create one compact overall comparison across all questions and metrics.
def evaluation_success_rate(evaluation_df):
    valid_success_values = evaluation_df["success"].dropna()
    if valid_success_values.empty:
        return None
    return valid_success_values.astype(bool).mean()


overall_prompt_comparison_df = pd.DataFrame([
    {
        "prompt": "Baseline",
        "average_score": baseline_evaluation_df["score"].mean(),
        "success_rate": evaluation_success_rate(
            baseline_evaluation_df
        ),
        "evaluated_scores": baseline_evaluation_df["score"].count(),
        "error_count": baseline_evaluation_df["error"].notna().sum(),
    },
    {
        "prompt": "Optimized",
        "average_score": optimized_evaluation_df["score"].mean(),
        "success_rate": evaluation_success_rate(
            optimized_evaluation_df
        ),
        "evaluated_scores": optimized_evaluation_df["score"].count(),
        "error_count": optimized_evaluation_df["error"].notna().sum(),
    },
])

print("Overall prompt comparison")
display(overall_prompt_comparison_df.round(4))

def normalize_prompt(text):
    return " ".join(text.split())

prompt_was_modified = (
    normalize_prompt(optimized_prompt_text)
    != normalize_prompt(seed_prompt_text)
)
optimized_metric_wins = (
    prompt_comparison_df["winner"] == "Optimized"
).sum()
baseline_metric_wins = (
    prompt_comparison_df["winner"] == "Baseline"
).sum()
metric_ties = (prompt_comparison_df["winner"] == "Tie").sum()

print(f"Did GEPA modify the seed prompt? {prompt_was_modified}")
print(
    "Metric wins — "
    f"Optimized: {optimized_metric_wins}, "
    f"Baseline: {baseline_metric_wins}, "
    f"Ties: {metric_ties}"
)

# Weighted scoring includes only metrics the generation prompt can influence.
# Context Relevance is reported separately in the retrieval evaluation.
METRIC_WEIGHTS = {
    "Answer Correctness": 0.40,
    "Faithfulness / Groundedness": 0.30,
    "Answer Relevance": 0.10,
    "Broker Actionability": 0.20,
}

def weighted_metric_score(summary_df):
    scores = summary_df.set_index("metric")["average_score"]
    return sum(scores.get(metric, 0) * weight for metric, weight in METRIC_WEIGHTS.items())

baseline_weighted = weighted_metric_score(baseline_metric_summary_df)
optimized_weighted = weighted_metric_score(optimized_metric_summary_df)
baseline_scores = baseline_metric_summary_df.set_index("metric")["average_score"]
optimized_scores = optimized_metric_summary_df.set_index("metric")["average_score"]
baseline_success_rate = evaluation_success_rate(baseline_evaluation_df)
optimized_success_rate = evaluation_success_rate(optimized_evaluation_df)

QUALITY_POLICY = {"minimum_gain": 0.01, "critical_tolerance": 0.02}
quality_gates = {
    "weighted_score_gain": optimized_weighted >= baseline_weighted + QUALITY_POLICY["minimum_gain"],
    "correctness_non_regression": optimized_scores["Answer Correctness"] >= baseline_scores["Answer Correctness"] - QUALITY_POLICY["critical_tolerance"],
    "faithfulness_non_regression": optimized_scores["Faithfulness / Groundedness"] >= baseline_scores["Faithfulness / Groundedness"] - QUALITY_POLICY["critical_tolerance"],
    "success_rate_non_regression": optimized_success_rate >= baseline_success_rate,
    "no_additional_errors": optimized_evaluation_df["error"].notna().sum() <= baseline_evaluation_df["error"].notna().sum(),
}
prompt_decision = "APPROVE" if all(quality_gates.values()) else "REJECT"
quality_gate_df = pd.DataFrame([
    {"gate": gate, "passed": passed}
    for gate, passed in quality_gates.items()
])

print(f"Weighted score — Baseline: {baseline_weighted:.4f} | Optimized: {optimized_weighted:.4f}")
print(f"Final prompt decision: {prompt_decision}")
display(quality_gate_df)


# Section 3: RAG Configuration Tuning and Evaluation


## 3.1 Define the RAG configurations

In [ ]:
# Objective: compare a small set of complete RAG configurations. Chunking
# parameters require a new index, while retrieval and generation parameters
# are applied at runtime. These are holistic candidates for selecting an
# overall approach, not a one-variable-at-a-time causal experiment.
rag_configurations = [
    {
        "name": "baseline",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
        "search_type": "similarity",
        "use_routing": True,
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 500,
    },
    {
        "name": "focused",
        "chunk_size": 700,
        "chunk_overlap": 100,
        "k": 4,
        "search_type": "similarity",
        "use_routing": True,
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 450,
    },
    {
        "name": "diverse",
        "chunk_size": 1200,
        "chunk_overlap": 250,
        "k": 6,
        "search_type": "mmr",
        "use_routing": True,
        "temperature": 0.1,
        "top_p": 0.9,
        "max_tokens": 650,
    },
]

# Use the optimized prompt only when it passed the Section 2.4 quality gates.
selected_prompt_for_tuning = (
    optimized_prompt if prompt_decision == "APPROVE" else optimized_prompt_seed
)
selected_prompt_name = (
    "optimized" if prompt_decision == "APPROVE" else "seed"
)

configuration_table_df = pd.DataFrame(rag_configurations)
print(f"Prompt used for RAG tuning: {selected_prompt_name}")
display(configuration_table_df)

## 3.2 Create a runtime answering function

In [ ]:
# Objective: build the index required by a configuration and answer one
# question with its retrieval and generation settings. Vector stores are
# cached in memory so each configuration is embedded only once per run.
configuration_vector_stores = {}
configuration_clients = {}


def build_configuration_vector_store(config):
    config_name = config["name"]
    if config_name in configuration_vector_stores:
        return configuration_vector_stores[config_name]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )
    config_sec_docs = splitter.split_documents(sec_raw_docs)

    for chunk_id, doc in enumerate(config_sec_docs):
        doc.metadata["chunk_id"] = chunk_id

    config_docs = csv_docs + config_sec_docs
    config_ids = [deterministic_document_id(doc) for doc in config_docs]

    client = chromadb.EphemeralClient()
    store = Chroma(
        client=client,
        collection_name=f"rag_tuning_{config_name}",
        embedding_function=embeddings,
    )
    store.add_documents(config_docs, ids=config_ids)

    configuration_clients[config_name] = client
    configuration_vector_stores[config_name] = store
    print(
        f"Built {config_name}: {len(csv_docs)} CSV documents + "
        f"{len(config_sec_docs)} SEC chunks"
    )
    return store


def retrieve_for_configuration(question, config):
    store = build_configuration_vector_store(config)
    datasets = (
        route_question(question)
        if config["use_routing"]
        else ["global_news", "stock_price_details", "sec_filings"]
    )
    constraints = extract_question_constraints(question)
    k_per_dataset = (
        config["k"]
        if len(datasets) == 1
        else max(2, config["k"] // len(datasets))
    )
    requested_date = (
        constraints["dates"][0]
        if len(constraints["dates"]) == 1
        else None
    )

    retrieved_docs = []
    searches = []

    for dataset in datasets:
        ticker_targets = (
            constraints["tickers"]
            if dataset == "stock_price_details" and constraints["tickers"]
            else [None]
        )
        k_per_target = max(1, k_per_dataset // len(ticker_targets))

        for ticker in ticker_targets:
            where = chroma_filter(
                dataset, ticker=ticker, date=requested_date
            )

            if config["search_type"] == "mmr":
                docs = store.max_marginal_relevance_search(
                    question,
                    k=k_per_target,
                    fetch_k=max(20, k_per_target * 4),
                    filter=where,
                )
            else:
                docs = store.similarity_search(
                    question, k=k_per_target, filter=where
                )

            fallback_used = False
            if not docs and dataset == "stock_price_details" and requested_date:
                fallback_used = True
                where = chroma_filter(dataset, ticker=ticker)
                docs = store.similarity_search(
                    question, k=k_per_target, filter=where
                )

            retrieved_docs.extend(docs)
            searches.append({
                "dataset": dataset,
                "ticker": ticker,
                "filter": where,
                "result_count": len(docs),
                "fallback_used": fallback_used,
            })

    unique_docs = []
    seen_ids = set()
    for doc in retrieved_docs:
        doc_id = deterministic_document_id(doc)
        if doc_id not in seen_ids:
            seen_ids.add(doc_id)
            unique_docs.append(doc)

    diagnostics = {
        "datasets_requested": datasets,
        "constraints": constraints,
        "search_type": config["search_type"],
        "searches": searches,
        "retrieved_count": len(unique_docs),
        "retrieved_datasets": sorted({
            doc.metadata["dataset"] for doc in unique_docs
        }),
    }
    return {"documents": unique_docs, "diagnostics": diagnostics}


def answer_with_configuration(question, config):
    retrieval_result = retrieve_for_configuration(question, config)
    docs = retrieval_result["documents"]
    formatted_context = format_retrieved_context(docs)

    rendered_prompt = selected_prompt_for_tuning.interpolate(
        question=question,
        context=formatted_context,
    )
    llm = ChatOpenAI(
        model="gpt-4.1-mini",
        temperature=config["temperature"],
        top_p=config["top_p"],
        max_tokens=config["max_tokens"],
    )
    response = llm.invoke([
        SystemMessage(content=FINANCIAL_SAFETY_POLICY),
        HumanMessage(content=rendered_prompt),
    ])

    return {
        "answer": response.content,
        "documents": docs,
        "formatted_context": formatted_context,
        "retrieval_context": [doc.page_content for doc in docs],
        "retrieved_sources": [
            format_source_metadata(doc.metadata or {}) for doc in docs
        ],
        "diagnostics": retrieval_result["diagnostics"],
    }

## 3.3 Evaluate one configuration on a benchmark set

In [ ]:
# Objective: evaluate one complete RAG configuration on a benchmark. Retrieval
# is intentionally rerun because it is the variable being tuned in Section 3.
def evaluate_rag_configuration(config, benchmark_df):
    answer_rows = []
    evaluation_rows = []

    for case_number, (_, row) in enumerate(benchmark_df.iterrows(), start=1):
        question = row["question"]
        expected_answer = row["response"]
        print(
            f"{config['name']}: case {case_number}/{len(benchmark_df)}"
        )

        result = answer_with_configuration(question, config)
        retrieval_checks = retrieval_quality(
            row,
            {
                "documents": result["documents"],
                "retrieved_sources": result["retrieved_sources"],
            },
        )

        test_case = LLMTestCase(
            input=question,
            actual_output=result["answer"],
            expected_output=expected_answer,
            retrieval_context=result["retrieval_context"],
        )

        answer_rows.append({
            "config_name": config["name"],
            "case_id": int(row["case_id"]),
            "question": question,
            "expected_output": expected_answer,
            "actual_output": result["answer"],
            "retrieved_sources": result["retrieved_sources"],
            "retrieved_count": len(result["documents"]),
            "retrieval_diagnostics": result["diagnostics"],
            **retrieval_checks,
        })

        # Context Relevance belongs here because each configuration can retrieve
        # different evidence. The four generation metrics evaluate the answer.
        for metric in rag_evaluation_metrics:
            evaluation_rows.append({
                "config_name": config["name"],
                "case_id": int(row["case_id"]),
                "question": question,
                "metric": metric.name,
                "coverage_status": retrieval_checks["coverage_status"],
                "source_match": retrieval_checks["source_match"],
                **measure_metric(metric, test_case),
            })

    return pd.DataFrame(evaluation_rows), pd.DataFrame(answer_rows)

## 3.4 Run all configurations on the test set

In [ ]:
# Objective: run every configuration on the same benchmark cases. After using
# this holdout to select a configuration, treat it as validation data rather
# than as an untouched final test set.
tuning_benchmark_df = evaluation_df.copy()
configuration_evaluation_frames = []
configuration_answer_frames = []

for config in rag_configurations:
    print(f"\nEvaluating RAG configuration: {config['name']}")
    config_evaluation_df, config_answers_df = evaluate_rag_configuration(
        config, tuning_benchmark_df
    )
    configuration_evaluation_frames.append(config_evaluation_df)
    configuration_answer_frames.append(config_answers_df)

all_config_evaluations_df = pd.concat(
    configuration_evaluation_frames, ignore_index=True
)
all_config_answers_df = pd.concat(
    configuration_answer_frames, ignore_index=True
)

print(
    f"Completed {len(rag_configurations)} configurations across "
    f"{len(tuning_benchmark_df)} benchmark cases."
)

## 3.5 Compare configurations

In [ ]:
# Objective: compare retrieval and generation quality together because both
# change across RAG configurations. The weights express project priorities.
configuration_metric_summary_df = (
    all_config_evaluations_df.groupby(["config_name", "metric"])
    .agg(
        average_score=("score", "mean"),
        success_rate=("success", "mean"),
        evaluated_cases=("score", "count"),
        error_count=("error", lambda values: values.notna().sum()),
    )
    .reset_index()
)

configuration_metric_scores_df = configuration_metric_summary_df.pivot(
    index="config_name",
    columns="metric",
    values="average_score",
).reset_index()

configuration_execution_summary_df = (
    all_config_evaluations_df.groupby("config_name")
    .agg(
        overall_average_score=("score", "mean"),
        overall_success_rate=("success", "mean"),
        error_count=("error", lambda values: values.notna().sum()),
    )
    .reset_index()
)

configuration_retrieval_summary_df = (
    all_config_answers_df.groupby("config_name")
    .agg(
        source_match_rate=("source_match", "mean"),
        supported_rate=(
            "coverage_status",
            lambda values: values.eq("SUPPORTED").mean(),
        ),
        average_retrieved_documents=("retrieved_count", "mean"),
    )
    .reset_index()
)

configuration_scorecard_df = (
    configuration_metric_scores_df
    .merge(configuration_execution_summary_df, on="config_name")
    .merge(configuration_retrieval_summary_df, on="config_name")
    .merge(
        configuration_table_df.rename(columns={"name": "config_name"}),
        on="config_name",
    )
)

RAG_TUNING_WEIGHTS = {
    "Answer Correctness": 0.35,
    "Faithfulness / Groundedness": 0.25,
    "Context Relevance": 0.20,
    "Answer Relevance": 0.05,
    "Broker Actionability": 0.15,
}

missing_metrics = set(RAG_TUNING_WEIGHTS).difference(
    configuration_scorecard_df.columns
)
if missing_metrics:
    raise ValueError(f"Missing tuning metrics: {sorted(missing_metrics)}")

configuration_scorecard_df["weighted_score"] = sum(
    configuration_scorecard_df[metric] * weight
    for metric, weight in RAG_TUNING_WEIGHTS.items()
)

configuration_scorecard_df = configuration_scorecard_df.sort_values(
    ["weighted_score", "source_match_rate"],
    ascending=False,
).reset_index(drop=True)

print("RAG configuration scorecard")
display(configuration_scorecard_df.round(4))

## 3.6 Select the best overall approach

In [ ]:
# Objective: select the strongest configuration without allowing a high
# weighted average to hide weak correctness, grounding, or execution errors.
eligible_configurations_df = configuration_scorecard_df[
    (configuration_scorecard_df["Answer Correctness"] >= EVALUATION_THRESHOLD)
    & (
        configuration_scorecard_df["Faithfulness / Groundedness"]
        >= EVALUATION_THRESHOLD
    )
    & (configuration_scorecard_df["error_count"] == 0)
]

if eligible_configurations_df.empty:
    selection_pool_df = configuration_scorecard_df
    rag_selection_decision = "SELECTED_WITH_WARNINGS"
else:
    selection_pool_df = eligible_configurations_df
    rag_selection_decision = "SELECTED"

best_configuration_row = selection_pool_df.sort_values(
    ["weighted_score", "source_match_rate"],
    ascending=False,
).iloc[0]

best_rag_config_name = best_configuration_row["config_name"]
best_rag_config = next(
    config
    for config in rag_configurations
    if config["name"] == best_rag_config_name
)

best_overall_approach = {
    "prompt": selected_prompt_name,
    "rag_configuration": best_rag_config,
    "weighted_score": float(best_configuration_row["weighted_score"]),
    "decision": rag_selection_decision,
}

print(f"Best RAG configuration: {best_rag_config_name}")
print(f"Selection decision: {rag_selection_decision}")
print(f"Weighted score: {best_overall_approach['weighted_score']:.4f}")
display(best_configuration_row.to_frame(name="value"))

## 3.7 Save the tuning results


In [ ]:
# Objective: persist derived tuning artifacts outside data/raw so the
# experiment can be reviewed without changing the source datasets.
tuning_output_dir = Path("../data/eval")
if not tuning_output_dir.exists():
    tuning_output_dir = Path("data/eval")
tuning_output_dir.mkdir(parents=True, exist_ok=True)

evaluation_path = tuning_output_dir / "rag_tuning_evaluations.csv"
answers_path = tuning_output_dir / "rag_tuning_answers.csv"
metric_summary_path = tuning_output_dir / "rag_tuning_metric_summary.csv"
scorecard_path = tuning_output_dir / "rag_tuning_scorecard.csv"
selection_path = tuning_output_dir / "best_rag_configuration.json"

all_config_evaluations_df.to_csv(evaluation_path, index=False)
all_config_answers_df.to_csv(answers_path, index=False)
configuration_metric_summary_df.to_csv(metric_summary_path, index=False)
configuration_scorecard_df.to_csv(scorecard_path, index=False)

selection_payload = {
    **best_overall_approach,
    "prompt_text": selected_prompt_for_tuning.text_template,
    "metric_weights": RAG_TUNING_WEIGHTS,
    "benchmark_case_ids": [
        int(case_id) for case_id in tuning_benchmark_df["case_id"].tolist()
    ],
}
with selection_path.open("w", encoding="utf-8") as file:
    json.dump(selection_payload, file, indent=2)

print("Saved RAG tuning artifacts:")
for path in [
    evaluation_path,
    answers_path,
    metric_summary_path,
    scorecard_path,
    selection_path,
]:
    print(f"- {path.resolve()}")

# Section 4: Final RAG Inference Pipeline with the Optimized Prompt and Best RAG Configuration

## 4.1 Define the final inference helper

In [ ]:
# Objective: expose one final inference function that uses the prompt approved
# in Section 2 and the best RAG configuration selected in Section 3.
required_final_objects = [
    "best_rag_config",
    "selected_prompt_for_tuning",
    "answer_with_configuration",
    "CORPUS_AS_OF_DATE",
    "FINANCIAL_SAFETY_POLICY",
]
missing_final_objects = [
    name for name in required_final_objects if name not in globals()
]
if missing_final_objects:
    raise RuntimeError(
        "Run Sections 1 through 3 before Section 4. Missing: "
        f"{missing_final_objects}"
    )

def final_rag_answer(question):
    result = answer_with_configuration(question, best_rag_config)
    return {
        "question": question,
        "answer": result["answer"],
        "source_documents": result["documents"],
        "sources": result["retrieved_sources"],
        "retrieval_diagnostics": result["diagnostics"],
        "config_name": best_rag_config["name"],
        "prompt_name": selected_prompt_name,
        "selection_decision": rag_selection_decision,
        "corpus_as_of_date": CORPUS_AS_OF_DATE,
    }

print(
    f"Final pipeline ready: prompt={selected_prompt_name}, "
    f"configuration={best_rag_config['name']}, "
    f"corpus_as_of={CORPUS_AS_OF_DATE}"
)

## 4.2 Define the final test cases

In [ ]:
# Objective: preserve the five official project test questions verbatim.
# Additional fields support deterministic retrieval diagnostics in Section 4.3;
# they do not alter the questions or act as reference answers.
final_test_cases = [
    {
        "id": 1,
        "question": (
            "Compare the total revenue and net income reported by Apple, Amazon, "
            "and Alphabet in their latest 10-Q filings. Which company grew fastest "
            "year-over-year?"
        ),
        "expected_datasets": ["sec_filings"],
        "required_entities": ["Apple", "Amazon", "Alphabet"],
        "evaluation_focus": "multi-company filing coverage and abstention",
    },
    {
        "id": 2,
        "question": (
            "Based on this week's news sentiment around Amazon, would you classify "
            "the current signal as bullish or bearish, and does the price data support that?"
        ),
        "expected_datasets": ["global_news", "stock_price_details"],
        "required_entities": ["Amazon"],
        "evaluation_focus": "cross-source sentiment and price grounding",
    },
    {
        "id": 3,
        "question": (
            "Should I invest in Apple right now? I heard Tim Cook is stepping down — "
            "is that a red flag? How's the stock been doing, and is there anything I should "
            "be worried about with the company's financials or risks?"
        ),
        "expected_datasets": [
            "global_news", "stock_price_details", "sec_filings"
        ],
        "required_entities": ["Apple", "Tim Cook"],
        "evaluation_focus": "rumor verification, grounding, and safe framing",
    },
    {
        "id": 4,
        "question": (
            "Is now a good time to buy Bitcoin? News is saying it just crossed $78k — "
            "am I too late? What's the price trend looked like over the last few months, "
            "and what's driving the current rally?"
        ),
        "expected_datasets": ["global_news", "stock_price_details"],
        "required_entities": ["Bitcoin"],
        "evaluation_focus": "claim verification and temporal evidence",
    },
    {
        "id": 5,
        "question": (
            "I want to invest in AI. Out of Google, Amazon, and Apple, which one looks "
            "like the best bet right now? How are they performing, what are they saying "
            "about their AI business, and what's the buzz around them?"
        ),
        "expected_datasets": [
            "global_news", "stock_price_details", "sec_filings"
        ],
        "required_entities": ["Alphabet", "Amazon", "Apple"],
        "evaluation_focus": "multi-source comparison without unsupported advice",
    },
]

if len(final_test_cases) != 5:
    raise ValueError("The project rubric requires exactly five final test cases.")

display(pd.DataFrame(final_test_cases)[
    ["id", "expected_datasets", "required_entities", "evaluation_focus"]
])

## 4.3 Run the final test cases

In [ ]:
# Objective: run qualitative final cases and make retrieval coverage visible.
# These cases have no reference answers, so the notebook does not claim an
# Answer Correctness score.
final_result_rows = []

for test_case in final_test_cases:
    print(f"\n{'=' * 80}")
    print(f"Final case {test_case['id']}: {test_case['evaluation_focus']}")

    try:
        result = final_rag_answer(test_case["question"])
        docs = result["source_documents"]
        retrieved_datasets = sorted({
            doc.metadata.get("dataset", "unknown") for doc in docs
        })
        missing_datasets = sorted(
            set(test_case["expected_datasets"]) - set(retrieved_datasets)
        )

        searchable_text = " ".join(
            [doc.page_content for doc in docs]
            + [
                " ".join(str(value) for value in (doc.metadata or {}).values())
                for doc in docs
            ]
        ).lower()
        missing_entities = [
            entity
            for entity in test_case["required_entities"]
            if entity.lower() not in searchable_text
        ]

        if not docs:
            coverage_status = "UNSUPPORTED"
        elif missing_datasets or missing_entities:
            coverage_status = "PARTIAL"
        else:
            coverage_status = "SUPPORTED"

        final_result_rows.append({
            "id": test_case["id"],
            "question": test_case["question"],
            "evaluation_focus": test_case["evaluation_focus"],
            "answer": result["answer"],
            "sources": result["sources"],
            "source_documents": docs,
            "retrieval_context": [doc.page_content for doc in docs],
            "retrieved_datasets": retrieved_datasets,
            "missing_datasets": missing_datasets,
            "missing_entities": missing_entities,
            "coverage_status": coverage_status,
            "config_name": result["config_name"],
            "prompt_name": result["prompt_name"],
            "retrieval_diagnostics": result["retrieval_diagnostics"],
            "error": None,
        })

        print("Question:")
        print(test_case["question"])
        print("\nAnswer:")
        print(result["answer"])
        print("\nSources:")
        for source_number, source in enumerate(result["sources"], start=1):
            print(f"[{source_number}] {source}")
        print(f"\nRetrieval coverage: {coverage_status}")
        if missing_datasets:
            print(f"Missing datasets: {missing_datasets}")
        if missing_entities:
            print(f"Missing entities: {missing_entities}")

    except Exception as exc:
        final_result_rows.append({
            "id": test_case["id"],
            "question": test_case["question"],
            "evaluation_focus": test_case["evaluation_focus"],
            "answer": None,
            "sources": [],
            "source_documents": [],
            "retrieval_context": [],
            "retrieved_datasets": [],
            "missing_datasets": test_case["expected_datasets"],
            "missing_entities": test_case["required_entities"],
            "coverage_status": "ERROR",
            "config_name": best_rag_config["name"],
            "prompt_name": selected_prompt_name,
            "retrieval_diagnostics": None,
            "error": str(exc),
        })
        print(f"Execution error: {exc}")

final_results_df = pd.DataFrame(final_result_rows)

print("\nFinal qualitative test summary")
display(final_results_df[[
    "id",
    "evaluation_focus",
    "coverage_status",
    "retrieved_datasets",
    "missing_datasets",
    "missing_entities",
    "config_name",
    "prompt_name",
    "error",
]])

# Run the same five official cases with the baseline prompt and every RAG
# configuration. Reuse the selected-configuration responses generated above
# so the rubric matrix contains exactly 5 x 4 distinct pipeline executions.
def official_case_coverage(test_case, docs):
    retrieved_datasets = sorted({
        doc.metadata.get("dataset", "unknown") for doc in docs
    })
    missing_datasets = sorted(
        set(test_case["expected_datasets"]) - set(retrieved_datasets)
    )
    searchable_text = " ".join(
        [doc.page_content for doc in docs]
        + [
            " ".join(str(value) for value in (doc.metadata or {}).values())
            for doc in docs
        ]
    ).lower()
    missing_entities = [
        entity
        for entity in test_case["required_entities"]
        if entity.lower() not in searchable_text
    ]
    if not docs:
        status = "UNSUPPORTED"
    elif missing_datasets or missing_entities:
        status = "PARTIAL"
    else:
        status = "SUPPORTED"
    return {
        "retrieved_datasets": retrieved_datasets,
        "missing_datasets": missing_datasets,
        "missing_entities": missing_entities,
        "coverage_status": status,
    }


official_answer_rows = []
selected_pipeline_name = f"optimized_prompt__{best_rag_config['name']}"
for row in final_result_rows:
    official_answer_rows.append({
        **row,
        "pipeline": selected_pipeline_name,
        "pipeline_type": "configuration",
    })

additional_pipelines = [
    {
        "pipeline": "baseline_prompt__baseline_retriever",
        "pipeline_type": "baseline",
        "prompt_name": "baseline",
        "config_name": "baseline_retriever",
        "config": None,
    }
] + [
    {
        "pipeline": f"optimized_prompt__{config['name']}",
        "pipeline_type": "configuration",
        "prompt_name": selected_prompt_name,
        "config_name": config["name"],
        "config": config,
    }
    for config in rag_configurations
    if config["name"] != best_rag_config["name"]
]

for pipeline_spec in additional_pipelines:
    print(f"\nRunning official cases: {pipeline_spec['pipeline']}")
    for test_case in final_test_cases:
        try:
            if pipeline_spec["pipeline_type"] == "baseline":
                result = baseline_rag_answer(test_case["question"])
                docs = result["source_documents"]
                answer = result["answer"]
                sources = [
                    format_source_metadata(doc.metadata or {}) for doc in docs
                ]
                diagnostics = result["retrieval_diagnostics"]
            else:
                result = answer_with_configuration(
                    test_case["question"], pipeline_spec["config"]
                )
                docs = result["documents"]
                answer = result["answer"]
                sources = result["retrieved_sources"]
                diagnostics = result["diagnostics"]

            coverage = official_case_coverage(test_case, docs)
            official_answer_rows.append({
                "id": test_case["id"],
                "question": test_case["question"],
                "evaluation_focus": test_case["evaluation_focus"],
                "answer": answer,
                "sources": sources,
                "source_documents": docs,
                "retrieval_context": [doc.page_content for doc in docs],
                **coverage,
                "config_name": pipeline_spec["config_name"],
                "prompt_name": pipeline_spec["prompt_name"],
                "retrieval_diagnostics": diagnostics,
                "pipeline": pipeline_spec["pipeline"],
                "pipeline_type": pipeline_spec["pipeline_type"],
                "error": None,
            })
        except Exception as exc:
            official_answer_rows.append({
                "id": test_case["id"],
                "question": test_case["question"],
                "evaluation_focus": test_case["evaluation_focus"],
                "answer": None,
                "sources": [],
                "source_documents": [],
                "retrieval_context": [],
                "retrieved_datasets": [],
                "missing_datasets": test_case["expected_datasets"],
                "missing_entities": test_case["required_entities"],
                "coverage_status": "ERROR",
                "config_name": pipeline_spec["config_name"],
                "prompt_name": pipeline_spec["prompt_name"],
                "retrieval_diagnostics": None,
                "pipeline": pipeline_spec["pipeline"],
                "pipeline_type": pipeline_spec["pipeline_type"],
                "error": str(exc),
            })

official_test_answers_df = pd.DataFrame(official_answer_rows)
expected_answer_runs = len(final_test_cases) * (1 + len(rag_configurations))
if len(official_test_answers_df) != expected_answer_runs:
    raise ValueError(
        f"Expected {expected_answer_runs} official-case answers, "
        f"received {len(official_test_answers_df)}."
    )

# Formally evaluate every official-case response with the four reference-free
# metrics. Answer Correctness remains excluded because no official reference
# answers were provided with these five questions.
official_evaluation_rows = []
for _, answer_row in official_test_answers_df.iterrows():
    test_case = LLMTestCase(
        input=answer_row["question"],
        actual_output=answer_row["answer"] or "",
        retrieval_context=answer_row["retrieval_context"],
    )
    for metric in official_test_evaluation_metrics:
        if answer_row["error"] is not None:
            measurement = {
                "score": None,
                "success": False,
                "reason": None,
                "error": answer_row["error"],
            }
        else:
            measurement = measure_metric(metric, test_case)
        official_evaluation_rows.append({
            "pipeline": answer_row["pipeline"],
            "pipeline_type": answer_row["pipeline_type"],
            "prompt_name": answer_row["prompt_name"],
            "config_name": answer_row["config_name"],
            "case_id": int(answer_row["id"]),
            "metric": metric.name,
            "coverage_status": answer_row["coverage_status"],
            **measurement,
        })

official_test_evaluation_df = pd.DataFrame(official_evaluation_rows)
expected_metric_runs = expected_answer_runs * len(official_test_evaluation_metrics)
if len(official_test_evaluation_df) != expected_metric_runs:
    raise ValueError(
        f"Expected {expected_metric_runs} official metric results, "
        f"received {len(official_test_evaluation_df)}."
    )

official_metric_summary_df = (
    official_test_evaluation_df.groupby(["pipeline", "metric"])
    .agg(
        average_score=("score", "mean"),
        success_rate=("success", "mean"),
        evaluated_cases=("score", "count"),
        failed_cases=("success", lambda values: values.eq(False).sum()),
        error_count=("error", lambda values: values.notna().sum()),
    )
    .reset_index()
)
official_scorecard_df = official_metric_summary_df.pivot(
    index="pipeline", columns="metric", values="average_score"
).reset_index()
official_coverage_summary_df = (
    official_test_answers_df.groupby(["pipeline", "coverage_status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("\nAll official-case answers (baseline plus three configurations)")
display(official_test_answers_df[[
    "pipeline", "id", "question", "answer",
    "coverage_status", "retrieved_datasets", "error",
]])

print("\nOfficial five-case execution coverage")
display(official_coverage_summary_df)
print("Official five-case formal evaluation")
display(official_metric_summary_df.round(4))
print("Official five-case scorecard")
display(official_scorecard_df.round(4))

# Save derived evaluation artifacts without modifying any raw source file.
official_output_dir = Path("../data/eval") if Path("../data").exists() else Path("data/eval")
official_output_dir.mkdir(parents=True, exist_ok=True)
official_test_answers_df.drop(
    columns=["source_documents", "retrieval_context"]
).to_csv(official_output_dir / "official_test_answers.csv", index=False)
official_test_evaluation_df.to_csv(
    official_output_dir / "official_test_evaluations.csv", index=False
)
official_scorecard_df.to_csv(
    official_output_dir / "official_test_scorecard.csv", index=False
)
print(f"Saved official test artifacts to: {official_output_dir.resolve()}")

# Conclusion

## Actionable Insights:

- WRITE INSIGHTS HERE

## Recommendations:

- WRITE RECOMMENDATIONS HERE

<font size=6>Power Ahead!</font>
___